In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import shap

file_path = r"C:\Users\brend\OneDrive - Stonehill College\Swing_Data.xlsx"
df = pd.read_excel(file_path)

In [55]:
df['side_encoded'] = df['side'].map({'L': 0, 'R': 1})

df.rename(columns={
    'avg_bat_speed': 'bat_speed',
    'swing_tilt': 'vertical_tilt',
    'avg_swing_length': 'swing_length',
    'avg_intercept_y_vs_batter': 'contact_point_rel_y'
}, inplace=True)

In [56]:
df['attack_direction_mirrored'] = df['attack_direction']
df.loc[df['side_encoded'] == 1, 'attack_direction_mirrored'] *= -1
df.loc[df['side_encoded'] == 0, 'attack_direction_mirrored'] *= -1
df['norm_attack_direction_mirrored'] = (
    df['attack_direction_mirrored'] / df['attack_direction_mirrored'].abs().max()
)

df['tilt_angle_interaction'] = df['vertical_tilt'] * df['attack_angle']
df['swing_aggressiveness'] = df['bat_speed'] * df['swing_length']
df['vertical_reach'] = df['attack_angle'] - df['contact_point_rel_y']
df['tilt_length_ratio'] = df['vertical_tilt'] / df['swing_length']

In [57]:
feature_cols = [
    'bat_speed', 'vertical_tilt', 'attack_angle', 'attack_direction_mirrored',
    'swing_length', 'contact_point_rel_y', 'tilt_angle_interaction',
    'norm_attack_direction_mirrored', 'swing_aggressiveness', 'vertical_reach',
    'tilt_length_ratio', 'side_encoded'
]

X = df[feature_cols]
y = df['xwobacon']

In [58]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

preds = np.zeros(len(df))
models = []
feature_importances = []
shap_interactions_list = []

for train_index, test_index in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    weights_train = df.iloc[train_index]['competitive_swings']

    model = XGBRegressor(
        n_estimators=1000,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train, sample_weight=weights_train)

    preds[test_index] = model.predict(X_test)

    models.append(model)
    feature_importances.append(model.feature_importances_)

    explainer = shap.TreeExplainer(model)
    shap_interactions = explainer.shap_interaction_values(X_test)
    shap_interactions_list.append(shap_interactions.mean(axis=0))

df['pxwobacon_model_oof'] = preds

In [62]:
def recompute_engineered_features(row):
    row['tilt_angle_interaction'] = row['vertical_tilt'] * row['attack_angle']
    row['swing_aggressiveness'] = row['bat_speed'] * row['swing_length']
    row['vertical_reach'] = row['attack_angle'] - row['contact_point_rel_y']
    row['tilt_length_ratio'] = row['vertical_tilt'] / row['swing_length']

    return row


def predict_pxwobacon(row, feature_cols, scaler, models):
    X = pd.DataFrame([row[feature_cols]], columns=feature_cols)
    X_scaled = scaler.transform(X)

    preds = np.array([model.predict(X_scaled)[0] for model in models])
    return preds.mean()


def pxwobacon_to_dSwing(pxwoba, side, side_params):

    mean_pred = side_params[side]["mean"]
    std_pred = side_params[side]["std"]

    return 100 + 10 * (pxwoba - mean_pred) / std_pred

In [ ]:
side_params = {}

for side in [0, 1]:
    mask = df['side_encoded'] == side
    mean_pred = preds[mask].mean()
    std_pred = preds[mask].std()

    side_params[side] = {
        "mean": mean_pred,
        "std": std_pred
    }

    df.loc[mask, 'dSwing+'] = 100 + 10 * (preds[mask] - mean_pred) / std_pred

In [ ]:
df['pxwobacon_scaled'] = np.nan

for side in [0, 1]:
    mask = df['side_encoded'] == side
    mean_pred = side_params[side]["mean"]
    std_pred = side_params[side]["std"]

    df.loc[mask, 'pxwobacon_scaled'] = (
        mean_pred
        + (df.loc[mask, 'dSwing+'] - 100) / 10 * std_pred
    )


df['pxwobacon_scaled'] = df['pxwobacon_scaled'].round(3)

In [ ]:
output_file = r"C:\Users\brend\OneDrive - Stonehill College\damage_swing_plus_results.xlsx"
df_output = df[[
    'year',
    'Team',
    'name',
    'dSwing+',
    'xwobacon',
    'pxwobacon_scaled'
]]
df_output.to_excel(output_file, index=False)
print(f"dSwing+ calculations complete! Results saved to {output_file}")

In [ ]:
teams = ['BOS', 'NYM', 'NYY']
df_2025 = df[(df['year'] == 2025) & (df['Team'].isin(teams))]

feature_cols = [
    'bat_speed', 'vertical_tilt', 'attack_angle', 'attack_direction_mirrored',
    'swing_length', 'contact_point_rel_y', 'tilt_angle_interaction',
    'norm_attack_direction_mirrored', 'swing_aggressiveness', 'vertical_reach',
    'tilt_length_ratio', 'side_encoded'
]

export_cols = ['year', 'Team', 'name', 'dSwing+'] + feature_cols

top_bottom_list = []

for team in teams:
    team_df = df_2025[df_2025['Team'] == team].copy()
    
    top3 = team_df.nlargest(3, 'dSwing+')
    bottom3 = team_df.nsmallest(3, 'dSwing+')
    
    top_bottom_list.append(top3)
    top_bottom_list.append(bottom3)

result_df = pd.concat(top_bottom_list)[export_cols]

output_file = r"C:\Users\brend\OneDrive - Stonehill College\swing_plus_top_bottom_2025_features.xlsx"
result_df.to_excel(output_file, index=False)

print(f"Top and bottom 3 dSwing+ players for BOS, NYM, NYY in 2025 exported to {output_file}")

In [ ]:
def simulate_metric_change_ensemble(
    df,
    year,
    player_name,
    metric_changes,
    feature_cols,
    scaler,
    models,
    side_params
):

    player_mask = (df['year'] == year) & (df['name'] == player_name)
    if player_mask.sum() == 0:
        raise ValueError("Player not found for that year.")
    
    player = df.loc[player_mask].iloc[0].copy()

    player = recompute_engineered_features(player)
    
    X_baseline = player[feature_cols].values.reshape(1, -1)
    X_baseline_scaled = scaler.transform(X_baseline)
    baseline_preds = np.array([model.predict(X_baseline_scaled)[0] for model in models])
    baseline_pxwoba = baseline_preds.mean()
    
    baseline_dSwing = pxwobacon_to_dSwing(
        baseline_pxwoba,
        player['side_encoded'],
        side_params
    )
    
    for metric, delta in metric_changes.items():
        if metric not in player:
            raise ValueError(f"Metric '{metric}' not found in dataframe.")
        player[metric] += delta
    
    player = recompute_engineered_features(player)
    
    X_new = player[feature_cols].values.reshape(1, -1)
    X_new_scaled = scaler.transform(X_new)
    new_preds = np.array([model.predict(X_new_scaled)[0] for model in models])
    new_pxwoba = new_preds.mean()
    
    side_std = side_params[player['side_encoded']]['std']
    delta_dSwing = 10 * (new_pxwoba - baseline_pxwoba) / side_std
    new_dSwing = baseline_dSwing + delta_dSwing
    
    result_table = pd.DataFrame([{
        "Year": year,
        "Player": player_name,
        "Baseline pxwOBAcon (model)": round(baseline_pxwoba, 3),
        "New pxwOBAcon (model)": round(new_pxwoba, 3),
        "Δ pxwOBAcon (model)": round(new_pxwoba - baseline_pxwoba, 4),
        "Baseline dSwing+": round(baseline_dSwing, 1),
        "New dSwing+": round(new_dSwing, 1),
        "Δ dSwing+": round(delta_dSwing, 2),
        "Metric Changes": metric_changes
    }])
    
    return result_table

In [61]:
result_table = simulate_metric_change_ensemble(
    df=df,
    year=2025,
    player_name="Mayer, Marcelo",
    metric_changes={
        'bat_speed': 1
    },
    feature_cols=feature_cols,
    scaler=scaler,
    models=models,
    side_params=side_params
)

print(result_table)

   Year          Player  Baseline pxwOBAcon  New pxwOBAcon  Δ pxwOBAcon  \
0  2025  Mayer, Marcelo                0.38          0.396        0.016   

   Baseline dSwing+  New dSwing+  Δ dSwing+    Metric Changes  
0             100.5        103.7       3.26  {'bat_speed': 1}  


C:\Users\brend\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\brend\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
